# Lesson 13: Protocol Features — Progress, Cancellation, URI Templates & Tasks

In this notebook, you will:
1. Emit `notifications/progress` updates from an async Python worker.
2. Handle cooperative `asyncio.CancelledError` cleanup.
3. Register RFC 6570 Resource Templates (`users://{user_id}/profile`) in `FastMCP`.

In [ ]:
!pip install -q -U mcp

In [ ]:
import asyncio
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ProtocolFeaturesDemo")

@mcp.resource("users://{user_id}/profile")
def user_profile(user_id: str) -> str:
    """Dynamic RFC 6570 user profile resource."""
    return f"{{"user_id": "{user_id}", "tier": "pro"}}"

async def cancellable_job():
    try:
        for i in range(1, 6):
            print(f"Progress: {i * 20}%")
            await asyncio.sleep(0.05)
    except asyncio.CancelledError:
        print("Cleanup executed cleanly on cancellation!")
        raise

task = asyncio.create_task(cancellable_job())
await asyncio.sleep(0.12)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("Verified cooperative cancellation.")

### Challenge Exercise
Build a `TaskManager` class supporting `create_task()`, `tasks_get(task_id)`, and `tasks_cancel(task_id)` that mirrors the `SEP-2663` Tasks extension.